# Telecom Customer Churn Analytics — Walkthrough
**Business question:** Why are customers leaving, how much revenue does it cost, and which customers should the retention team call first?

**Data:** IBM Telco Customer Churn sample — 7,043 customers, 21 attributes (demographics, services, contract, billing, churn flag).
Pipeline: SQL (DuckDB) for cleaning and KPI tables → Python (SciPy, scikit-learn) for statistical tests and the predictive model.

In [1]:
import duckdb, pandas as pd, json
pd.set_option('display.width', 160)
con = duckdb.connect('../data/churn.duckdb', read_only=True)
q = lambda s: con.sql(s).df()
T = '../outputs/tables/'

## 1. Data quality

In [2]:
q('SELECT * FROM data_quality_log')

,check_name,value
0,Customers,7043
1,Duplicate customer IDs,0
2,Blank TotalCharges (tenure 0) → 0,11
3,Negative charges,0


## 2. Headline KPIs: customer churn vs revenue churn

In [3]:
q('SELECT * FROM headline_kpis').T

,0
customers,7043.00
churned_customers,1869.00
churn_rate_pct,26.50
total_mrr,456117.00
churned_mrr,139131.00
mrr_churn_pct,30.50
annualised_revenue_lost,1669570.00
avg_bill_churners,74.44
avg_bill_retained,61.27
avg_tenure_churners,18.00


Churners pay a higher monthly bill on average, so **revenue churn (≈30% of MRR) is higher than customer churn (26.5%)**.

## 3. Churn by driver (SQL)

In [4]:
q("SELECT * FROM churn_by_dimension WHERE dimension IN ('Contract','Tenure band','Internet service','Payment method')")

,dimension,value,customers,churned,churn_rate_pct,churned_mrr,share_of_churners_pct
0,Contract,Month-to-month,3875,1655.0,42.7,120847.0,88.6
1,Contract,One year,1473,166.0,11.3,14118.0,8.9
2,Contract,Two year,1695,48.0,2.8,4165.0,2.6
3,Internet service,Fiber optic,3096,1297.0,41.9,114300.0,69.4
4,Internet service,DSL,2421,459.0,19.0,22529.0,24.6
5,Internet service,No,1526,113.0,7.4,2302.0,6.0
6,Payment method,Electronic check,2365,1071.0,45.3,84289.0,57.3
7,Payment method,Mailed check,1612,308.0,19.1,16804.0,16.5
8,Payment method,Bank transfer (automatic),1544,258.0,16.7,20092.0,13.8
9,Payment method,Credit card (automatic),1522,232.0,15.2,17947.0,12.4


![drivers](../outputs/charts/01_churn_by_driver.png)
![matrix](../outputs/charts/02_risk_matrix.png)

In [5]:
q('SELECT * FROM top_risk_segments')

,contract,internet_service,payment_method,customers,churned,churn_rate_pct,churned_mrr
0,Month-to-month,Fiber optic,Electronic check,1307,789.0,60.4,68282.0
1,Month-to-month,Fiber optic,Mailed check,201,102.0,50.7,8407.0
2,Month-to-month,Fiber optic,Bank transfer (automatic),327,149.0,45.6,13062.0
3,Month-to-month,Fiber optic,Credit card (automatic),293,122.0,41.6,10731.0
4,Month-to-month,DSL,Electronic check,474,192.0,40.5,8776.0
5,Month-to-month,DSL,Mailed check,367,113.0,30.8,5292.0
6,Month-to-month,DSL,Credit card (automatic),185,50.0,27.0,2475.0
7,One year,Fiber optic,Electronic check,196,51.0,26.0,5150.0
8,Month-to-month,No,Mailed check,325,67.0,20.6,1345.0
9,Month-to-month,DSL,Bank transfer (automatic),197,39.0,19.8,1824.0


## 4. Are the differences statistically significant?
Chi-square tests (categorical) with Cramér's V effect size; Mann-Whitney U for tenure and monthly charges (rank-biserial effect size).

In [6]:
pd.read_csv(T + 'driver_significance_tests.csv')

,feature,test,statistic,p_value,effect_size,significant_at_5pct
0,tenure_months,Mann-Whitney U,2515538.0,2.420000e-208,0.480,True
1,contract,chi-square,1184.6,5.860000e-258,0.410,True
2,OnlineSecurity,chi-square,850.0,2.660000e-185,0.347,True
3,TechSupport,chi-square,828.2,1.440000e-180,0.343,True
4,InternetService,chi-square,732.3,9.570000e-160,0.322,True
5,payment_method,chi-square,648.1,3.680000e-140,0.303,True
6,OnlineBackup,chi-square,601.8,2.080000e-131,0.292,True
7,DeviceProtection,chi-square,558.4,5.510000e-122,0.282,True
8,monthly_charges,Mann-Whitney U,6003126.0,3.310000e-54,-0.242,True
9,StreamingTV,chi-square,374.2,5.530000e-82,0.231,True


Tenure, contract, online security, tech support and internet type have the largest effects. Gender is **not** significant, so it is not a useful targeting criterion.

## 5. Predictive model
Logistic regression (interpretable) vs gradient boosting, stratified 75/25 split + 5-fold CV.

In [7]:
display(pd.read_csv(T + 'model_performance.csv'))
json.loads(open(T + 'model_summary.json').read())

,model,cv_auc_mean,cv_auc_std,test_auc,precision,recall,f1
0,Logistic Regression,0.844,0.015,0.846,0.521,0.797,0.630
1,Gradient Boosting,0.845,0.013,0.846,0.665,0.501,0.571


{'best_model': 'Logistic Regression',
 'test_auc': 0.846,
 'top3_deciles_capture': 66.8,
 'top_decile_lift': 2.83,
 'confusion': {'tn': 952, 'fp': 342, 'fn': 95, 'tp': 372}}

In [8]:
pd.read_csv(T + 'logistic_odds_ratios.csv')

,feature,coefficient,odds_ratio
0,contract_Month-to-month,0.627,1.87
1,InternetService_Fiber optic,0.503,1.65
2,total_charges,0.474,1.61
3,OnlineSecurity_No,0.214,1.24
4,payment_method_Electronic check,0.203,1.23
5,TechSupport_No,0.183,1.20
6,paperless_billing,0.183,1.20
7,StreamingMovies_Yes,0.160,1.17
8,StreamingTV_Yes,0.117,1.12
9,addon_count,0.098,1.10


![odds](../outputs/charts/03_odds_ratios.png)

## 6. Turning the model into a retention plan

In [9]:
display(pd.read_csv(T + 'gains_by_decile.csv'))
display(pd.read_csv(T + 'active_customers_by_risk_tier.csv'))
pd.read_csv(T + 'retention_target_list_top500.csv').head(10)

,decile,customers,churners,avg_score,churn_rate_pct,cum_pct_churners_captured,lift
0,1,177,133,0.873,75.1,28.5,2.83
1,2,177,105,0.795,59.3,51.0,2.24
2,3,177,74,0.706,41.8,66.8,1.58
3,4,177,59,0.587,33.3,79.4,1.26
4,5,177,41,0.444,23.2,88.2,0.87
5,6,177,26,0.322,14.7,93.8,0.55
6,7,177,18,0.204,10.2,97.6,0.38
7,8,177,8,0.115,4.5,99.4,0.17
8,9,177,2,0.054,1.1,99.8,0.04
9,10,168,1,0.017,0.6,100.0,0.02


,risk_tier,customers,mrr
0,Low,2833,155422.0
1,Medium,1315,82651.0
2,High,1026,78913.0


,customer_id,contract,tenure_months,InternetService,payment_method,monthly_charges,churn_risk_score,risk_tier
0,5150-ITWWB,Month-to-month,3,Fiber optic,Electronic check,94.85,0.932,High
1,2545-EBUPK,Month-to-month,2,Fiber optic,Electronic check,84.05,0.921,High
2,6630-UJZMY,Month-to-month,4,Fiber optic,Electronic check,83.25,0.914,High
3,4847-TAJYI,Month-to-month,6,Fiber optic,Electronic check,89.35,0.912,High
4,6350-XFYGW,Month-to-month,4,Fiber optic,Electronic check,94.75,0.910,High
5,8161-QYMTT,Month-to-month,7,Fiber optic,Electronic check,94.10,0.909,High
6,4115-NZRKS,Month-to-month,7,Fiber optic,Electronic check,89.15,0.908,High
7,3320-VEOYC,Month-to-month,14,Fiber optic,Electronic check,95.60,0.908,High
8,0021-IKXGC,Month-to-month,1,Fiber optic,Electronic check,72.10,0.907,High
9,0187-QSXOE,Month-to-month,7,Fiber optic,Electronic check,89.00,0.906,High


![gains](../outputs/charts/04_gains_curve.png)

## 7. Recommendations
1. **Contract migration:** offer discounted 1-year upgrades to month-to-month customers in the first 6 months (55% churn in this cell).
2. **Fix the fiber experience:** fiber customers churn at 42% vs 19% on DSL despite paying more. Investigate pricing and service quality.
3. **Move e-check payers to autopay:** electronic-check customers churn at 45% vs 15–17% on automatic payments.
4. **Bundle tech support and online security:** each is linked to roughly a third of the churn rate.
5. **Use the risk score:** call the top 30% of customers first to reach ~67% of likely churners. The list is exported to `outputs/tables/retention_target_list_top500.csv`.